# Notebook 00 - Coverage Check (M2.1.3)

Determines which cities pass the air-quality inclusion rule.

**Rule:** City needs >= 4 valid Kaggle years (2015-2019) AND >= 1 valid CPCB year (2023-2024).

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.utils.helpers import setup_logging
setup_logging()
sns.set_theme(style="whitegrid")
FIGURES = "../reports/figures"
os.makedirs(FIGURES, exist_ok=True)
print("Environment ready.")

## 1. Kaggle coverage check

In [ ]:
from src.loaders.load_air_kaggle import run_coverage_check as kaggle_cov_fn
from src.utils.constants import RAW_AIR_KAGGLE
print(f'Folder: {RAW_AIR_KAGGLE}')
print(f'Files: {list(RAW_AIR_KAGGLE.glob("*.csv"))}')

In [ ]:
try:
    kaggle_cov = kaggle_cov_fn()
    print(f'Cities: {len(kaggle_cov)}')
    display(kaggle_cov)
except FileNotFoundError as e:
    print(f'Missing: {e}')
    print('Place city_day.csv in data/raw/air_kaggle/')
    kaggle_cov = pd.DataFrame()

## 2. CPCB coverage check

In [ ]:
from src.loaders.load_air_cpcb import run_coverage_check as cpcb_cov_fn
from src.utils.constants import RAW_AIR_CPCB
print(f'Folder: {RAW_AIR_CPCB}')
try:
    cpcb_cov = cpcb_cov_fn()
    print(f'Cities: {len(cpcb_cov)}')
    display(cpcb_cov)
except Exception as e:
    print(f'CPCB failed: {e}')
    print('Place AQS_2023.pdf in data/raw/air_cpcb/')
    cpcb_cov = pd.DataFrame()

## 3. Combined table

In [ ]:
if not kaggle_cov.empty and not cpcb_cov.empty:
    combined = kaggle_cov.merge(cpcb_cov, on='city', how='outer')
    combined['passes_both'] = combined.get('passes_kaggle_rule', False) & combined.get('passes_cpcb_rule', False)
elif not kaggle_cov.empty:
    combined = kaggle_cov.copy(); combined['passes_both'] = combined.get('passes_kaggle_rule', False)
else:
    combined = pd.DataFrame()
if not combined.empty: display(combined)

In [ ]:
if not combined.empty and 'kaggle_valid_year_count' in combined.columns:
    fig, ax = plt.subplots(figsize=(10,6))
    top = combined.nlargest(20, 'kaggle_valid_year_count')
    colors = ['#27ae60' if v else '#e74c3c' for v in top.get('passes_kaggle_rule', [False]*len(top))]
    ax.barh(top['city'], top['kaggle_valid_year_count'], color=colors)
    ax.axvline(4, color='k', ls='--', label='Threshold (4 yrs)')
    ax.set_xlabel('Valid years in 2015-2019')
    ax.set_title('Kaggle Coverage (top 20 cities)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/00_coverage.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
os.makedirs('../data/interim', exist_ok=True)
if not combined.empty:
    combined.to_csv('../data/interim/city_coverage.csv', index=False)
    print('Saved: data/interim/city_coverage.csv')

## Next steps

1. Review `data/interim/city_coverage.csv`
2. Update `data/processed/places.csv` if needed
3. Update `data/raw/name_aliases.csv`
4. Run `01_data_profiling.ipynb`